# FAF level Food Outflow Prediction GNN Model 1.0


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, GCNConv
import torch.nn.init as init
import matplotlib.pyplot as plt
from model import GCN, GAT
from utils import train, test

code = '1'
node_df = pd.read_csv("data/county_aligned_filtered.csv")
edges_df = pd.read_csv(f"data/county_flows_with_ports_distance_gravity.csv")


In [2]:
edges_df['dms_orig_cnty'] = edges_df['dms_orig_cnty'].astype(str).str.zfill(5)
edges_df['dms_dest_cnty'] = edges_df['dms_dest_cnty'].astype(str).str.zfill(5)
edges_df.head()



,dms_orig_cnty,dms_dest_cnty,tons_2022,distance,ports_0,ports_1,log_distance,inv_distance,gravity_index,log_gravity_index
0,01005,01005,0,0.000000,1,0,0.000000,1.000000,6.328746e+07,17.963198
1,01005,01023,0,270.581901,1,0,5.604264,0.003682,4.441114e+02,6.098325
2,01005,01035,0,158.898440,1,0,5.074539,0.006254,1.238585e+03,7.122532
3,01005,01051,0,107.657281,1,0,4.688199,0.009203,1.767314e+04,9.779858
4,01005,01065,0,231.875724,1,0,5.450505,0.004294,6.925312e+02,6.541796


In [3]:
print(len(edges_df['dms_orig_cnty'].unique()))
print(len(edges_df['dms_dest_cnty'].unique()))

3142
3142


In [4]:
len(edges_df)

9872164

In [5]:
# Drop rows where FIPS is '00nan'
node_df = node_df[node_df['FIPS'] != '00nan']


In [6]:
len(node_df['FIPS'].unique())

3117

In [7]:
len(edges_df['dms_orig_cnty'].unique())

3142

In [8]:
excluded_states = ['60', '66', '69', '72', '78']
# Ensure FIPS codes are 5 digits by padding with zeros
node_df['FIPS'] = node_df['FIPS'].astype(str).str.zfill(5)


In [9]:
len(node_df['FIPS'].unique())

3117

In [10]:
model_path = f"models/best_model{code}_gcn.pth"


In [11]:
node_df.head()

,FIPS,Poverty Percent,------,11----,21----,22----,23----,31----,42----,44----,...,net_SCTG_35_share,net_SCTG_36_share,net_SCTG_37_share,net_SCTG_38_share,net_SCTG_39-43_share,net_SCTG_39_share,net_SCTG_40_share,net_SCTG_41_share,net_SCTG_43_share,population
0,01001,13.4,11036.0,93.0,60.0,176.0,453.0,920.0,251.0,2634.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,55390.0
1,01003,10.1,61792.0,10.0,93.0,266.0,3571.0,4160.0,2719.0,13378.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,212521.0
2,01005,33.4,6834.0,149.0,83.0,0.0,89.0,2500.0,121.0,856.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25157.0
3,01007,20.2,3484.0,74.0,0.0,19.0,863.0,327.0,90.0,502.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22550.0
4,01009,12.8,6645.0,7.0,0.0,0.0,514.0,1059.0,404.0,1207.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,57787.0


In [12]:
import geopandas as gpd
ports = gpd.read_file("data/shapefiles/Principal_Ports.geojson")

In [13]:
county = gpd.read_file('data/shapefiles/cb_2017_us_county_500k/cb_2017_us_county_500k.shp')

In [14]:
excluded_states = ['60', '66', '69', '72', '78']
filtered_counties = county[~county['GEOID'].str[:2].isin(excluded_states)]
print(f"Number of unique counties after excluding states {excluded_states}: {len(filtered_counties['GEOID'].unique())}")

Number of unique counties after excluding states ['60', '66', '69', '72', '78']: 3142


In [15]:
# Get counties in filtered_counties but not in node_df
missing_counties = set(filtered_counties['GEOID']) - set(node_df['FIPS'])
print(f"Number of counties missing from node_df: {len(missing_counties)}")
print("\nFirst 10 missing counties:")
print(list(missing_counties))




Number of counties missing from node_df: 26

First 10 missing counties:
['18087', '08014', '24035', '46102', '02220', '02158', '18091', '19141', '35011', '18033', '35013', '17099', '02020', '02282', '02230', '02068', '12086', '24037', '02110', '02195', '24033', '02275', '02105', '22059', '42083', '02198']


In [16]:
# Convert MH_Income from string to float by removing commas
node_df['MH_Income'] = node_df['MH_Income'].str.replace(',', '').astype(float)

# Check remaining columns with string data
print("Columns with string data:")
for col in node_df.columns:
    if node_df[col].dtype == 'object':
        print(f"{col}: {node_df[col].dtype}")
        print(f"Sample values: {node_df[col].head()}\n")

Columns with string data:
FIPS: object
Sample values: 0    01001
1    01003
2    01005
3    01007
4    01009
Name: FIPS, dtype: object



In [17]:
print(missing_counties)

{'18087', '08014', '24035', '46102', '02220', '02158', '18091', '19141', '35011', '18033', '35013', '17099', '02020', '02282', '02230', '02068', '12086', '24037', '02110', '02195', '24033', '02275', '02105', '22059', '42083', '02198'}


In [18]:
# Create a copy of node_df to avoid modifying the original
node_df_with_missing = node_df.copy()

# Get the mean values for each state (first two digits of FIPS)
state_means = node_df.groupby(node_df['FIPS'].str[:2]).mean().astype(float)


# Create new rows for missing counties
new_rows = []
for missing_county in missing_counties:
    print(missing_county)
    state_code = missing_county[:2]
    if state_code in state_means.index:
        # Create a new row with the state's mean values
        new_row = state_means.loc[state_code].copy()
        new_row['FIPS'] = missing_county
        new_rows.append(new_row)

# Add the new rows to the dataframe
if new_rows:
    node_df_with_missing = pd.concat([node_df_with_missing, pd.DataFrame(new_rows)], ignore_index=True)

# Update the original node_df
node_df = node_df_with_missing

print(f"Number of counties after adding missing ones: {len(node_df)}")


18087
08014
24035
46102
02220
02158
18091
19141
35011
18033
35013
17099
02020
02282
02230
02068
12086
24037
02110
02195
24033
02275
02105
22059
42083
02198
Number of counties after adding missing ones: 3143


In [19]:
# Get list of valid FIPS codes from node_df
valid_fips = node_df['FIPS'].unique()

# Filter edge_df to only include rows where both origin and destination counties are in valid_fips
edges_df = edges_df[
    (edges_df['dms_orig_cnty'].isin(valid_fips)) &
    (edges_df['dms_dest_cnty'].isin(valid_fips))
]

print(f"Number of edges after filtering: {len(edges_df)}")


Number of edges after filtering: 9872164


In [20]:
# Filter to only include mainland US counties (01-56, excluding territories)
node_df['state_fips'] = node_df['FIPS'].astype(str).str.zfill(5).str[:2]
valid_states = [f"{i:02d}" for i in range(1, 57)]
exclude = {'60', '66', '69', '72', '78'}  # Territories
valid_states = [s for s in valid_states if s not in exclude]
node_df = node_df[node_df['state_fips'].isin(valid_states)]
print(f"Number of counties after filtering: {len(node_df['FIPS'].unique())}")  # Should be 3144


Number of counties after filtering: 3143


In [21]:
print(len(edges_df))

9872164


In [22]:
edges_df.head()

,dms_orig_cnty,dms_dest_cnty,tons_2022,distance,ports_0,ports_1,log_distance,inv_distance,gravity_index,log_gravity_index
0,01005,01005,0,0.000000,1,0,0.000000,1.000000,6.328746e+07,17.963198
1,01005,01023,0,270.581901,1,0,5.604264,0.003682,4.441114e+02,6.098325
2,01005,01035,0,158.898440,1,0,5.074539,0.006254,1.238585e+03,7.122532
3,01005,01051,0,107.657281,1,0,4.688199,0.009203,1.767314e+04,9.779858
4,01005,01065,0,231.875724,1,0,5.450505,0.004294,6.925312e+02,6.541796


In [23]:
edges_df.head()

,dms_orig_cnty,dms_dest_cnty,tons_2022,distance,ports_0,ports_1,log_distance,inv_distance,gravity_index,log_gravity_index
0,01005,01005,0,0.000000,1,0,0.000000,1.000000,6.328746e+07,17.963198
1,01005,01023,0,270.581901,1,0,5.604264,0.003682,4.441114e+02,6.098325
2,01005,01035,0,158.898440,1,0,5.074539,0.006254,1.238585e+03,7.122532
3,01005,01051,0,107.657281,1,0,4.688199,0.009203,1.767314e+04,9.779858
4,01005,01065,0,231.875724,1,0,5.450505,0.004294,6.925312e+02,6.541796


In [24]:
# Create a mapping dictionary for FAF zones
node_df['FIPS'] = node_df['FIPS'].astype(str).str.zfill(5)
edges_df['dms_orig_cnty'] = edges_df['dms_orig_cnty'].astype(str).str.zfill(5)
edges_df['dms_dest_cnty'] = edges_df['dms_dest_cnty'].astype(str).str.zfill(5)

faf_zones = node_df['FIPS'].unique()
zone_to_idx = {zone: idx for idx, zone in enumerate(faf_zones)}
idx_to_zone = {idx: zone for idx, zone in enumerate(faf_zones)}
node_df['FIPS'] = node_df['FIPS'].map(zone_to_idx)

# Remap the origin and destination in edges_df
edges_df['dms_orig_cnty'] = edges_df['dms_orig_cnty'].map(zone_to_idx)
edges_df['dms_dest_cnty'] = edges_df['dms_dest_cnty'].map(zone_to_idx)

# Check the remapped data
print("\nEdges dataframe with remapped indices:")
print(edges_df.head())



Edges dataframe with remapped indices:
   dms_orig_cnty  dms_dest_cnty  tons_2022    distance  ports_0  ports_1  \
0              2              2          0    0.000000        1        0   
1              2             11          0  270.581901        1        0   
2              2             17          0  158.898440        1        0   
3              2             25          0  107.657281        1        0   
4              2             32          0  231.875724        1        0   

   log_distance  inv_distance  gravity_index  log_gravity_index  
0      0.000000      1.000000   6.328746e+07          17.963198  
1      5.604264      0.003682   4.441114e+02           6.098325  
2      5.074539      0.006254   1.238585e+03           7.122532  
3      4.688199      0.009203   1.767314e+04           9.779858  
4      5.450505      0.004294   6.925312e+02           6.541796  


In [25]:
node_df.head()

,FIPS,Poverty Percent,------,11----,21----,22----,23----,31----,42----,44----,...,net_SCTG_36_share,net_SCTG_37_share,net_SCTG_38_share,net_SCTG_39-43_share,net_SCTG_39_share,net_SCTG_40_share,net_SCTG_41_share,net_SCTG_43_share,population,state_fips
0,0,13.4,11036.0,93.0,60.0,176.0,453.0,920.0,251.0,2634.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,55390.0,01
1,1,10.1,61792.0,10.0,93.0,266.0,3571.0,4160.0,2719.0,13378.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,212521.0,01
2,2,33.4,6834.0,149.0,83.0,0.0,89.0,2500.0,121.0,856.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25157.0,01
3,3,20.2,3484.0,74.0,0.0,19.0,863.0,327.0,90.0,502.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22550.0,01
4,4,12.8,6645.0,7.0,0.0,0.0,514.0,1059.0,404.0,1207.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,57787.0,01


In [26]:
edges_df.head()

,dms_orig_cnty,dms_dest_cnty,tons_2022,distance,ports_0,ports_1,log_distance,inv_distance,gravity_index,log_gravity_index
0,2,2,0,0.000000,1,0,0.000000,1.000000,6.328746e+07,17.963198
1,2,11,0,270.581901,1,0,5.604264,0.003682,4.441114e+02,6.098325
2,2,17,0,158.898440,1,0,5.074539,0.006254,1.238585e+03,7.122532
3,2,25,0,107.657281,1,0,4.688199,0.009203,1.767314e+04,9.779858
4,2,32,0,231.875724,1,0,5.450505,0.004294,6.925312e+02,6.541796


In [27]:
# Transform the target columns
edges_df['tons_2022'] = np.log1p(edges_df['tons_2022'])




In [28]:
from sklearn.preprocessing import StandardScaler
features_scaler = StandardScaler()
edges_df['distance'] = features_scaler.fit_transform(edges_df.iloc[:, 3].values.reshape(-1, 1))
print(edges_df.head())

   dms_orig_cnty  dms_dest_cnty  tons_2022  distance  ports_0  ports_1  \
0              2              2        0.0 -1.486515        1        0   
1              2             11        0.0 -1.206972        1        0   
2              2             17        0.0 -1.322354        1        0   
3              2             25        0.0 -1.375292        1        0   
4              2             32        0.0 -1.246960        1        0   

   log_distance  inv_distance  gravity_index  log_gravity_index  
0      0.000000      1.000000   6.328746e+07          17.963198  
1      5.604264      0.003682   4.441114e+02           6.098325  
2      5.074539      0.006254   1.238585e+03           7.122532  
3      4.688199      0.009203   1.767314e+04           9.779858  
4      5.450505      0.004294   6.925312e+02           6.541796  


In [29]:
# Normalize the remaining edge features
edge_features = [ 'log_distance', 'inv_distance', 'gravity_index', 'log_gravity_index']
for feature in edge_features:
    edges_df[feature] = features_scaler.fit_transform(edges_df[feature].values.reshape(-1, 1))

print("Normalized edge features:")
print(edges_df[edge_features].head())


Normalized edge features:
   log_distance  inv_distance  gravity_index  log_gravity_index
0     -9.624916     55.797162       0.016654           5.804520
1     -1.968350      0.120675      -0.001055           0.769807
2     -2.692062      0.264395      -0.001055           1.204417
3     -3.219882      0.429207      -0.001051           2.332020
4     -2.178416      0.154875      -0.001055           0.957989


In [30]:
# PCA for feature reduction
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
features_scaler = StandardScaler()
node_df.iloc[:, 1:] = features_scaler.fit_transform(node_df.iloc[:, 1:])
node_df.head()

# Select the features to be reduced
features = node_df.drop(columns=['FIPS'])

# Apply PCA for feature reduction
pca = PCA(n_components=30)  # Adjust the number of components as needed
reduced_features = pca.fit_transform(features)

# Create a new DataFrame with the reduced features
reduced_df = pd.DataFrame(reduced_features, columns=[f'PC{i+1}' for i in range(reduced_features.shape[1])])
reduced_df = pd.concat([node_df['FIPS'], reduced_df], axis=1)

# Update node_df with the reduced features
node_df = reduced_df

# Verify the changes
print(node_df.head())

   FIPS       PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
0     0 -4.294404 -0.673664 -0.414376 -0.693009  0.290532 -1.117672 -0.476491   
1     1 -4.303351 -0.837589  1.757685 -0.811136 -0.209197 -0.776597 -0.718079   
2     2 -4.160011 -0.792810 -1.075129 -0.704876  0.730141 -1.941050 -0.696416   
3     3 -4.350643 -0.503349 -1.973179  0.571182  1.115170 -1.572940 -0.604184   
4     4 -4.380090 -0.518406  0.451425 -1.694465 -0.644628  0.279401 -1.030172   

        PC8       PC9  ...      PC21      PC22      PC23      PC24      PC25  \
0 -1.290285  0.064133  ...  2.284205 -0.431408  1.214891 -0.768999 -0.236958   
1 -1.527530 -0.382241  ...  1.684782 -0.374922  1.020269 -0.098551  0.111289   
2 -2.232732 -0.463361  ...  1.835935 -0.312108  1.486818 -0.342327 -1.552869   
3 -0.602516  0.510575  ...  0.372637 -0.307883  0.265512 -0.073877 -0.066860   
4 -1.285897  0.285418  ...  1.568426  0.103156  1.447149 -1.045436 -0.256717   

       PC26      PC27      PC28 

In [31]:


# Extract node features
node_features = []
for i, row in node_df.iterrows():
    node_features.append(torch.tensor(row.iloc[1:].values, dtype=torch.float))
node_features = torch.stack(node_features)
# Extract edge information (source, target)
edge_index = torch.tensor(edges_df[['dms_orig_cnty', 'dms_dest_cnty']].values.T, dtype=torch.long)

# Extract edge target values (food flow)
edge_y = torch.tensor(edges_df['tons_2022'].values, dtype=torch.float).view(-1, 1)

# Extract additional edge features
edge_attr = torch.tensor(edges_df[['distance', 'log_distance', 'inv_distance', 'gravity_index', 'log_gravity_index', 'ports_0', 'ports_1']].values, dtype=torch.float)  # Excluding source, target, food_flow

# Create PyG Data object
data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=edge_y)


In [32]:
edge_index.shape

torch.Size([2, 9872164])

In [33]:
edge_attr

tensor([[-1.4865e+00, -9.6249e+00,  5.5797e+01,  ...,  5.8045e+00,
          1.0000e+00,  0.0000e+00],
        [-1.2070e+00, -1.9684e+00,  1.2067e-01,  ...,  7.6981e-01,
          1.0000e+00,  0.0000e+00],
        [-1.3224e+00, -2.6921e+00,  2.6440e-01,  ...,  1.2044e+00,
          1.0000e+00,  0.0000e+00],
        ...,
        [-3.4803e-01, -5.3592e-02, -3.4427e-02,  ...,  1.2824e+00,
          0.0000e+00,  1.0000e+00],
        [-1.0878e+00, -1.4846e+00,  5.9316e-02,  ...,  1.1755e+00,
          0.0000e+00,  1.0000e+00],
        [-1.4865e+00, -9.6249e+00,  5.5797e+01,  ...,  7.6155e+00,
          0.0000e+00,  1.0000e+00]])

In [34]:
# Reshape edge_attr to be 2D if it's 1D
if len(data.edge_attr.shape) == 1:
    # Reshape to [num_edges, 1]
    data.edge_attr = data.edge_attr.view(-1, 1)
    edge_feature_dim = 1
else:
    edge_feature_dim = data.edge_attr.shape[1]

# Debug info
num_edges = data.edge_index.shape[1]
print(f"Dataset size: {num_edges} edges")
print(f"Data shapes - edge_index: {data.edge_index.shape}, y: {data.y.shape}")
print(f"Edge attribute shape: {data.edge_attr.shape}")

print(f"Edge feature dimension: {edge_feature_dim}")



Dataset size: 9872164 edges
Data shapes - edge_index: torch.Size([2, 9872164]), y: torch.Size([9872164, 1])
Edge attribute shape: torch.Size([9872164, 7])
Edge feature dimension: 7


In [35]:
# Divide the data into 100 batches of equal size for testing/debugging
total_edges = data.edge_index.shape[1]
batch_size = total_edges // 100  # Calculate how many edges per batch

# Create a list to store all batches
sliced_data = []

# Create 100 equally sized batches
for i in range(100):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if i < 99 else total_edges  # For the last batch, include remaining edges

    # Select indices for this batch
    batch_indices = torch.arange(start_idx, end_idx)

    # Create sliced data for this batch
    sliced_edge_index = data.edge_index[:, batch_indices]
    sliced_edge_attr = data.edge_attr[batch_indices]
    sliced_y = data.y[batch_indices]

    # Create a new Data object with the sliced components
    batch_data = Data(
        x=data.x,  # Keep all node features
        edge_index=sliced_edge_index,
        edge_attr=sliced_edge_attr,
        y=sliced_y
    )

    # Add to our list of batches
    sliced_data.append(batch_data)

# Keep the original data for reference
original_data = data

# Use the first batch for immediate testing
data = sliced_data[0]

print(f"Divided data into 100 batches of approximately {batch_size} edges each")
print(f"First batch shapes - edge_index: {data.edge_index.shape}, y: {data.y.shape}, edge_attr: {data.edge_attr.shape}")
print(f"Total batches: {len(sliced_data)}")


Divided data into 100 batches of approximately 98721 edges each
First batch shapes - edge_index: torch.Size([2, 98721]), y: torch.Size([98721, 1]), edge_attr: torch.Size([98721, 7])
Total batches: 100


In [36]:
gcn_model = GCN(in_channels=data.x.shape[1], hidden_channels=64, edge_feature_dim=edge_feature_dim)
gcn_model.load_state_dict(torch.load(model_path))



<All keys matched successfully>

In [37]:
from utils import inference_batch

In [38]:
results = []
for d in sliced_data[:50]:
    result = inference_batch(gcn_model, [d])
    results.append(result)


100%|██████████| 1/1 [00:00<00:00,  9.32it/s]


In [39]:
print(results[0][0])
# Merge all results into a single dataframe
for result in results:
    for key, value in result[0].items():
        if isinstance(value, torch.Tensor):
            result[0][key] = value.detach().numpy()
        else:
            result[0][key] = value



{'origin': tensor([  2,   2,   2,  ..., 300, 300, 300]), 'dest': tensor([   2,   11,   17,  ..., 1437, 1677, 1680]), 'exist_prob': tensor([0.9855, 0.5043, 0.4570,  ..., 0.6413, 0.4843, 0.4646]), 'exist_pred': tensor([1., 1., 0.,  ..., 1., 0., 0.]), 'value': tensor([9.8300, 0.7197, 0.8331,  ..., 2.2615, 1.1376, 0.8792])}


In [40]:
for result in results:
    origins = result[0]['origin']
    destinations = result[0]['dest']
    new_origins = []
    new_destinations = []
    for origin in origins:
        new_origins.append(idx_to_zone[origin])
    for destination in destinations:
        new_destinations.append(idx_to_zone[destination])
    result[0]['origin'] = new_origins
    result[0]['dest'] = new_destinations


In [41]:
for result in results:
    result[0]['exist'] = result[0]['exist_pred'].flatten()
    result[0]['value'] = result[0]['value'].flatten()
    result[0]['predicted_value_original'] = np.exp(result[0]['value']) - 1

In [42]:
df_list = []
for result in results:
    df = pd.DataFrame(result[0])
    df_list.append(df)
df = pd.concat(df_list)

df.to_csv(f'mjf_results/predicted_sctg_{code}_01.csv', index=False)

In [43]:
results = []
for d in sliced_data[50:100]:
    result = inference_batch(gcn_model, [d])
    results.append(result)

100%|██████████| 1/1 [00:00<00:00,  5.32it/s]


In [44]:
print(results[0][0])
# Merge all results into a single dataframe
for result in results:
    for key, value in result[0].items():
        if isinstance(value, torch.Tensor):
            result[0][key] = value.detach().numpy()
        else:
            result[0][key] = value



{'origin': tensor([2356, 2356, 2356,  ...,  461,  461,  461]), 'dest': tensor([2703, 2789, 2835,  ..., 1391, 1398, 1400]), 'exist_prob': tensor([0.3493, 0.3195, 0.4154,  ..., 0.6705, 0.6266, 0.6691]), 'exist_pred': tensor([0., 0., 0.,  ..., 1., 1., 1.]), 'value': tensor([0.1607, 0.2104, 0.2795,  ..., 1.7709, 0.7644, 1.6406])}


In [45]:
for result in results:
    origins = result[0]['origin']
    destinations = result[0]['dest']
    new_origins = []
    new_destinations = []
    for origin in origins:
        new_origins.append(idx_to_zone[origin])
    for destination in destinations:
        new_destinations.append(idx_to_zone[destination])
    result[0]['origin'] = new_origins
    result[0]['dest'] = new_destinations


In [46]:
for result in results:
    result[0]['exist'] = result[0]['exist_pred'].flatten()
    result[0]['value'] = result[0]['value'].flatten()
    result[0]['predicted_value_original'] = np.exp(result[0]['value']) - 1

In [47]:
df_list = []
for result in results:
    df = pd.DataFrame(result[0])
    df_list.append(df)
df = pd.concat(df_list)

df.to_csv(f'mjf3_results/predicted_sctg_{code}_02.csv', index=False)